# Recommandation d'animes basée sur le contenu

L'objectif de cette partie est de construire un moteur de recommandation basé sur le contenu qui calcule la similarité entre les animes en fonction de leurs genres (`anime.csv`).

In [4]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import html
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

import kagglehub
from kagglehub import KaggleDatasetAdapter

animes_df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "CooperUnion/anime-recommendations-database",
    "anime.csv"
)
ratings_df = kagglehub.dataset_load(
    KaggleDatasetAdapter.PANDAS,
    "CooperUnion/anime-recommendations-database",
    "rating.csv"
)

animes = animes_df[['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members']].copy()
ratings = ratings_df[['user_id', 'anime_id', 'rating']].copy()

animes['name'] = animes['name'].astype(str).apply(html.unescape)
animes.head()

,anime_id,name,genre,type,episodes,rating,members
0,32281,Kimi no Na wa.,"Drama, Romance, School, Supernatural",Movie,1,9.37,200630
1,5114,Fullmetal Alchemist: Brotherhood,"Action, Adventure, Drama, Fantasy, Magic, Mili...",TV,64,9.26,793665
2,28977,Gintama°,"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.25,114262
3,9253,Steins;Gate,"Sci-Fi, Thriller",TV,24,9.17,673572
4,9969,Gintama',"Action, Comedy, Historical, Parody, Samurai, S...",TV,51,9.16,151266


In [5]:
from sklearn.feature_extraction.text import TfidfVectorizer

# Remplacer les genres manquants par une chaîne vide
animes['genre'] = animes['genre'].fillna('').astype(str)

# Vectorisation TF-IDF sur les genres
tf = TfidfVectorizer(analyzer='word', ngram_range=(1, 2), min_df=1, stop_words='english')
tfidf_matrix = tf.fit_transform(animes['genre'])
print("Dimensions de la matrice TF-IDF :", tfidf_matrix.shape)

Dimensions de la matrice TF-IDF : (12294, 654)


In [6]:
from sklearn.metrics.pairwise import linear_kernel
cosine_sim = linear_kernel(tfidf_matrix, tfidf_matrix)
cosine_sim[:4, :4]

array([[1.        , 0.04304778, 0.        , 0.        ],
       [0.04304778, 1.        , 0.04968007, 0.        ],
       [0.        , 0.04968007, 1.        , 0.11713654],
       [0.        , 0.        , 0.11713654, 1.        ]])

In [8]:
titles = animes['name']
indices = pd.Series(animes.index, index=animes['name'])

def genre_recommendations(title):
    idx = indices[title]
    sim_scores = list(enumerate(cosine_sim[idx]))
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)
    sim_scores = sim_scores[1:21]
    anime_indices = [i[0] for i in sim_scores]
    return titles.iloc[anime_indices]